## Step 1: Scrape data from bina.az

Import libraries needed for scraping: requests to send HTTP calls
(since bina.az has no official API docs, we replicate the browser's
request), json to encode/decode GraphQL parameters, sys for UTF-8
output (Azerbaijani letters break Windows' default encoding), time
for delays between requests (avoid overloading the server), csv to
save results.

In [1]:
import requests
import json
# import sys
import time
import csv

# sys.stdout.reconfigure(encoding="utf-8")

Define the API endpoint and request headers, mimicking a real browser
request. The x-platform header is required — without it the server
returns 400 Bad Request (found by comparing our headers against a
real browser request in DevTools).

In [2]:
url = "https://bina.az/graphql"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Referer": "https://bina.az/",
    "x-platform": "desktop",
    "Content-Type": "application/json",
}

Parse one raw listing node into a flat dict with the fields we need,
since the raw JSON has deeply nested fields we don't want to repeat
everywhere. Uses .get() with fallback {} since some fields (e.g.
location) can be null for non-apartment listings (land, commercial).

In [3]:
def parse_node(node):
    price = node.get("price") or {}
    area = node.get("area") or {}
    location = node.get("location") or {}
    city = node.get("city") or {}
    return {
        "id": node.get("id"),
        "price": price.get("total"),
        "currency": price.get("currency"),
        "rooms": node.get("rooms"),
        "area": area.get("value"),
        "floor": node.get("floor"),
        "hasRepair": node.get("hasRepair"),
        "location": location.get("name"),
        "city": city.get("name"),
    }

Fetch one page of listings from the API. Cursor is used for
pagination — the site returns 24 listings per request, so we need
to request page by page to collect everything.

In [4]:
def fetch_page(cursor=None):
    variables = {"first": 24}
    if cursor:
        variables["cursor"] = cursor

    page_params = {
        "operationName": "FeaturedItemsRow",
        "variables": json.dumps(variables),
        "extensions": json.dumps({
            "persistedQuery": {
                "version": 1,
                "sha256Hash": "cc02557ea77b3a51bdca72328af5c60f34c8d80280918d98115862d009a0a31a"
            }
        }),
    }

    response = requests.get(url, params=page_params, headers=headers)
    return response.json()

Loop through all pages until hasNextPage is False, since we don't
know in advance how many pages exist. Collect parsed listings into
all_items, 1s pause between requests to be polite to the server.

In [5]:
all_items = []
cursor = None
page_number = 0

while True:
    result = fetch_page(cursor)

    if "data" not in result:
        print(result)
        break

    edges = result["data"]["featuredItems"]["edges"]
    page_info = result["data"]["featuredItems"]["pageInfo"]

    for edge in edges:
        all_items.append(parse_node(edge["node"]))

    print("page", page_number, "got", len(edges), "items")
    page_number += 1

    if not page_info["hasNextPage"]:
        break

    cursor = page_info["endCursor"]
    time.sleep(1)

print("total", len(all_items))

page 0 got 24 items
page 1 got 24 items
page 2 got 24 items
page 3 got 24 items
page 4 got 24 items
page 5 got 24 items
page 6 got 24 items
page 7 got 24 items
page 8 got 24 items
page 9 got 24 items
page 10 got 24 items
page 11 got 24 items
page 12 got 24 items
page 13 got 24 items
page 14 got 24 items
page 15 got 24 items
page 16 got 24 items
page 17 got 24 items
page 18 got 24 items
page 19 got 24 items
page 20 got 24 items
page 21 got 24 items
page 22 got 24 items
page 23 got 24 items
page 24 got 24 items
page 25 got 24 items
page 26 got 24 items
page 27 got 24 items
page 28 got 24 items
page 29 got 24 items
page 30 got 24 items
page 31 got 24 items
page 32 got 24 items
page 33 got 24 items
page 34 got 24 items
page 35 got 24 items
page 36 got 24 items
page 37 got 24 items
page 38 got 24 items
page 39 got 24 items
page 40 got 24 items
page 41 got 24 items
page 42 got 24 items
page 43 got 24 items
page 44 got 24 items
page 45 got 24 items
page 46 got 24 items
page 47 got 24 items
pa

Save collected listings to CSV so data persists after the script/kernel
finishes running — all_items only exists in memory otherwise.

In [6]:
with open("../data/items.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=all_items[0].keys())
    writer.writeheader()
    writer.writerows(all_items)

## Step 2: Clean the data

Import pandas for tabular data manipulation, used throughout cleaning
and feature preparation.

In [7]:
import pandas as pd

Read raw scraped data from CSV instead of using all_items directly,
so cleaning can be re-run independently without re-scraping the site.

In [8]:
df = pd.read_csv("../data/items.csv")
df.shape

(1350, 9)

Clean the data: drop rows with missing rooms/location (land, commercial
listings), remove price outliers (price >= 5000, price_per_m2 >= 300 to
filter out rental listings mixed into sale data), round price_per_m2.

In [9]:
df_clean = df[df["rooms"].notna()]
df_clean["rooms"] = df_clean["rooms"].astype(int)
df_clean = df_clean[df_clean["price"] >= 5000]
df_clean["price_per_m2"] = df_clean["price"] / df_clean["area"]
df_clean = df_clean[df_clean["price_per_m2"] >= 300]
df_clean["price_per_m2"] = df_clean["price_per_m2"].round(2)
df_clean = df_clean[df_clean["location"].notna()]

df_clean.shape

(954, 10)

Save cleaned data to CSV, decoupling the training step from cleaning —
train.ipynb's later cells can reload from item_clean.csv without
re-running the cleaning logic.

In [10]:
df_clean.to_csv("../data/item_clean.csv", index=False, encoding="utf-8")

## Step 3: Prepare features for the model

Group rare locations (< 5 listings) into "Other" — with 81 unique
locations and only 955 rows, most locations have too few examples
for the model to learn anything reliable from them individually.

In [11]:
location_counts = df_clean["location"].value_counts()
common_locations = location_counts[location_counts >= 5].index
df_clean["location"] = df_clean["location"].where(df_clean["location"].isin(common_locations), "Other")

df_clean["location"].nunique()

46

One-hot encode location: turn the single text column into N binary
columns (one per category), since linear/tree models need numeric
input and location has no natural numeric order.

In [12]:
df_encoded = pd.get_dummies(df_clean, columns=["location"])
df_encoded.shape

(954, 55)

Split into features (X) and target (y). Exclude id (random, no signal),
price and price_per_m2 (these ARE the target or derived from it —
including them would leak the answer), currency and city (single
value across ~all rows, no signal).

In [13]:
features = df_encoded.drop(columns=["id", "price", "currency", "price_per_m2", "city"])
target = df_encoded["price"]

features.shape, target.shape

((954, 50), (954,))

Fill missing floor with median (16.5% missing — too many rows to drop),
and missing hasRepair with the most common value (only 1 row missing).

In [14]:
features["floor"] = features["floor"].fillna(features["floor"].median())
features["hasRepair"] = features["hasRepair"].fillna(features["hasRepair"].mode()[0])

features.isna().sum().sum()

np.int64(0)

Split into train/test sets (80/20). Random split (not by row order)
avoids biased train/test sets — e.g. if data were sorted by price,
a non-random split could put mostly cheap listings in train and
expensive ones in test. random_state=42 fixes the shuffle for
reproducibility.

In [15]:
from sklearn.model_selection import train_test_split

features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

features_train.shape, features_test.shape

((763, 50), (191, 50))

## Step 4: Train and evaluate models

Baseline model: Linear Regression. Simple, fast, interpretable —
a reference point before trying more complex models.

In [16]:
features.isna().sum()[features.isna().sum() > 0]

Series([], dtype: int64)

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

model = LinearRegression()
model.fit(features_train, target_train)

predictions = model.predict(features_test)
mae = mean_absolute_error(target_test, predictions)
mae

85707.0749241852

Random Forest: captures non-linear relationships better than Linear
Regression (e.g. floor's effect on price may not be strictly linear).
Same train/test split and features as above for a fair comparison.

In [18]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(features_train, target_train)

rf_predictions = rf_model.predict(features_test)
rf_mae = mean_absolute_error(target_test, rf_predictions)
rf_mae

89527.46048782514

In [20]:
target_test.mean()

np.float64(385743.44502617803)

In [21]:
features.columns.tolist()

['rooms',
 'area',
 'floor',
 'hasRepair',
 'location_20 Yanvar',
 'location_28 May',
 'location_7-ci mikrorayon',
 'location_8 Noyabr',
 'location_Abşeron',
 'location_Avtovağzal',
 'location_Azadlıq Prospekti',
 'location_Ağ şəhər',
 'location_Badamdar',
 'location_Bakmil',
 'location_Bakıxanov',
 'location_Bayıl',
 'location_Bilgəh',
 'location_Biləcəri',
 'location_Binə',
 'location_Binəqədi',
 'location_Buzovna',
 'location_Dərnəgül',
 'location_Elmlər Akademiyası',
 'location_Gənclik',
 'location_Hövsan',
 'location_Həzi Aslanov',
 'location_Koroğlu',
 'location_Lökbatan',
 'location_Masazır',
 'location_Memar Əcəmi',
 'location_Mərdəkan',
 'location_Nardaran',
 'location_Neftçilər',
 'location_Nizami',
 'location_Novxanı',
 'location_Nəriman Nərimanov',
 'location_Nərimanov',
 'location_Nəsimi',
 'location_Other',
 'location_Qara Qarayev',
 'location_Sea Breeze',
 'location_Səbail',
 'location_Xətai',
 'location_Yasamal',
 'location_Yeni Günəşli',
 'location_İnşaatçılar',
 'loca

Linear Regression outperforms Random Forest on this feature set once
compared fairly (same features, same NaN handling) — save it as the
final model.

In [22]:
import joblib

joblib.dump(model, "../models/model.pkl")

['../models/model.pkl']